In [1]:
from faker import Faker
from datetime import datetime, timedelta
import random
import psycopg2
import psycopg2
from psycopg2.extras import execute_values

In [2]:
#=====CONFIG======
total_authors = 20
total_articles = 1000
total_views = 2000

start_date = datetime(2016, 1, 1)
end_date = datetime(2023, 1, 1)

#=====Seed=======
Faker.seed(50)
random.seed(50)
fake = Faker("id_ID")

In [3]:
def connection():
    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        dbname="kumparan_source",
        user="source_user",
        password="source_pass"
    )
    return conn

In [4]:
def categories(cur):
    lists = ["News", "Bisnis", "Tekno", "Bola", "Entertainment",
              "Otomotif", "Sains", "Food & Travel"]
    data = []
    for cat in lists:
        data.append((cat, start_date, end_date))
    execute_values(cur, "INSERT INTO categories (name, created_at, updated_at) VALUES %s", data)
    print(f"Inserted {len(data)} categories")

def authors(cur):
    last_join = end_date - timedelta(days=365)
    lists = ["News", "Bisnis", "Tekno", "Bola", "Entertainment",
                  "Otomotif", "Sains", "Food & Travel"]
    data = []
    for i in range(total_authors):
        name = fake.name()
        desk = random.choice(lists)
        join_date = fake.date_time_between(start_date, last_join)
        data.append((name, desk, join_date, join_date, join_date))
    execute_values(cur, "INSERT INTO authors (name, desk, joined_at, created_at, updated_at) VALUES %s", data)
    print(f"Inserted {len(data)} authors")

def create_one_article(author_rows):
    # author_rows isinya (id, joined_at) - artikel nggak boleh dibuat sebelum penulisnya join
    author_id, joined_at = random.choice(author_rows)
    created_at = fake.date_time_between(joined_at, end_date)
    title = fake.sentence(nb_words=8)
    content = fake.text(max_nb_chars=2000)

    #buat 10% artikel belum di publish / draft
    if random.random() < 0.1:
        published_at = None
    else:
        published_at = created_at + timedelta(minutes=random.randint(5,600))
        if published_at > end_date:
            published_at = end_date

    if published_at is not None:
        update_base = published_at
    else:
        update_base = created_at
    updated_at = update_base + timedelta(days=random.randint(0,30))
    if updated_at > end_date:
        updated_at = end_date

    #buat 3% artikel sudah soft deleted
    if random.random() < 0.03:
        deleted_at = updated_at
    else:
        deleted_at = None

    #buat 10% artikel kotor dengan tag html
    if random.random() < 0.1:
        content = f"<p>{content}</p>"

    return(title, content, published_at, author_id, created_at, updated_at, deleted_at)

def fill_articles(cur, conn):
    cur.execute("SELECT id, joined_at FROM authors")
    author_rows = cur.fetchall()

    #insert articles
    sql = """
        insert into articles (title, content, published_at, author_id, created_at, updated_at, deleted_at)
        values %s
    """
    batch = []
    for i in range(total_articles):
        batch.append(create_one_article(author_rows))
        if len(batch) >= 5000:
            execute_values(cur, sql, batch)
            conn.commit()
            print(f"Inserted {len(batch)} articles")
            batch = []

    if batch:
        execute_values(cur, sql, batch)
        conn.commit()
        print(f"Inserted {len(batch)} articles")

def fill_article_categories(cur):
    cur.execute("SELECT id FROM articles")
    article_ids = []
    rows = cur.fetchall()
    for row in rows:
        article_ids.append(row[0])

    cur.execute("SELECT id FROM categories")
    category_ids = []
    rows = cur.fetchall()
    for row in rows:
        category_ids.append(row[0])

    data = []
    for article_id in article_ids:
        category_id = random.choice(category_ids)
        data.append((article_id, category_id))
    execute_values(cur, "INSERT INTO article_categories (article_id, category_id) VALUES %s", data)
    print(f"Inserted {len(data)} article_categories")

def article_views(cur):
    cur.execute("select id, published_at from articles where published_at is not null")
    published_articles = cur.fetchall()
    device_list = ['android', 'ios', 'web', 'desktop']

    #insert part
    sql = "insert into article_views (article_id, device, viewed_at) values %s"   

    batch = []
    for i in range(total_views):
        article_id, published_at = random.choice(published_articles)
        viewed_at = published_at + timedelta(minutes=random.randint(1, 10000))
        if viewed_at > end_date:
            viewed_at = end_date

        batch.append((article_id, random.choice(device_list), viewed_at))
        if len(batch) >= 5000:
            execute_values(cur, sql, batch)
            print(f"Inserted {len(batch)} article_views")
            batch = []

    if batch:
        execute_values(cur, sql, batch)
        print(f"Inserted {len(batch)} article_views")

In [5]:
#===========MAIN==========
def main():
    conn = connection()
    cur = conn.cursor()

    categories(cur)
    authors(cur)
    conn.commit()

    fill_articles(cur, conn)
    fill_article_categories(cur)
    conn.commit()

    article_views(cur)
    conn.commit()
    cur.close()
    conn.close()
    print("Data generation completed successfully.")


if __name__ == "__main__":
    main()

Inserted 8 categories
Inserted 20 authors
Inserted 1000 articles
Inserted 1000 article_categories
Inserted 2000 article_views
Data generation completed successfully.
